# Работа с данными кодов IATA

Коды ИАТА (IATA — International Air Transport Association) — это индивидуальные идентификаторы объектов, имеющих значение для индустрии пассажирских авиаперевозок. Они присваиваются Международной ассоциацией воздушного транспорта.

Эти коды нужны для обращения к API "Яндекс Расписаний" для нахождения рейсов.

Для данной работы был использован датасет с Kaggle, который содержит более 7500 подобных наименований: https://www.kaggle.com/datasets/thoudamyoihenba/airports.

## Импорт и минимальный EDA

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from functools import lru_cache

tqdm.pandas()

In [2]:
df_iata = pd.read_csv('../data/Airports-Only.csv', encoding='latin1')
df_iata = df_iata[df_iata['IATA'].notna()][['Name', 'City', 'Country', 'IATA']].rename(columns={'Name': 'name_en', 'City': 'city_en', 'Country': 'country_en'}).reset_index(drop=True)
df_iata.head()

,name_en,city_en,country_en,IATA
0,Goroka Airport,Goroka,Papua New Guinea,GKA
1,Madang Airport,Madang,Papua New Guinea,MAG
2,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU
3,Nadzab Airport,Nadzab,Papua New Guinea,LAE
4,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM


In [3]:
df_iata.info()

<class 'pandas.DataFrame'>
RangeIndex: 7698 entries, 0 to 7697
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name_en     7698 non-null   str  
 1   city_en     7649 non-null   str  
 2   country_en  7698 non-null   str  
 3   IATA        7698 non-null   str  
dtypes: str(4)
memory usage: 240.7 KB


In [4]:
df_iata.isna().sum()

name_en        0
city_en       49
country_en     0
IATA           0
dtype: int64

Некоторые значения городов отсутствуют. При переводе на русский, в этих ячейках также будут значения `nan`

### Отсутствующие IATA коды

In [5]:
df_iata['IATA'].value_counts()

IATA
\N     1626
GKA       1
MAG       1
HGU       1
LAE       1
       ... 
UGU       1
ETM       1
MNH       1
CGY       1
CPO       1
Name: count, Length: 6073, dtype: int64

In [6]:
df_iata[df_iata['IATA'] == r'\N'].head()

,name_en,city_en,country_en,IATA
21,Winnipeg / St. Andrews Airport,Winnipeg,Canada,\N
22,Halifax / CFB Shearwater Heliport,Halifax,Canada,\N
43,Princeton Airport,Princeton,Canada,\N
103,Pitt Meadows Airport,Pitt Meadows,Canada,\N
210,Boufarik Airport,Boufarik,Algeria,\N


`IATA == \N` означает отсутствие кода у аэропорта (отсутствие в самом датасете или отсутствие по определению, например когда аэропорт - военная база). Не будет помещать их в общую БД

In [7]:
df_iata = df_iata[df_iata['IATA'] != r'\N'].reset_index(drop=True)

In [8]:
df_iata.shape

(6072, 4)

### Повторяющиеся наименования аэропортов

In [9]:
df_iata['name_en'].value_counts()

name_en
San Pedro Airport              3
Newcastle Airport              3
Santa Maria Airport            3
Capital City Airport           3
Deer Lake Airport              2
                              ..
Bilogai-Sugapa Airport         1
Ramon Airport                  1
Rustaq Airport                 1
Laguindingan Airport           1
Desierto de Atacama Airport    1
Name: count, Length: 6046, dtype: int64

In [10]:
df_iata[df_iata['name_en'] == 'San Pedro Airport']

,name_en,city_en,country_en,IATA
245,San Pedro Airport,San Pedro,Cote d'Ivoire,SPY
3393,San Pedro Airport,San Pedro,Belize,SPR
4770,San Pedro Airport,Bonanza,Nicaragua,BZA


В датасете присутствуют повторяющиеся названия, но у разных аэропортов с разными IATA кодами. Это может ввести агента в заблуждение при поиске кода IATA по названию аэропортов. Подобный сценарий необходимо продумать в системном промте и инструментах. А также протестировать в задачах.

## Перевод

In [11]:
import translators as ts


@lru_cache(maxsize=1000)
def translator_error_handle(phrase: str) -> str:
    result = None
    try:
        result = ts.translate_text(phrase,
                                   translator='yandex',
                                   from_language='en',
                                   to_language='ru')
        return result
    except Exception:
        return np.nan
    

In [12]:
# Тесты перевода
print(ts.translate_text('Mount Hagen Kagamuga Airport', translator='yandex', from_language='en', to_language='ru'))
print(ts.translate_text('Tolmachevo Airport', translator='yandex', from_language='en', to_language='ru'))
print(ts.translate_text('Hornafjörður Airport', translator='yandex', from_language='en', to_language='ru'))

Аэропорт Маунт-Хаген-Кагамуга
Аэропорт Толмачево
Аэропорт Хорнафьордур


In [13]:
print('Старт перевода названий аэропортов...')
df_iata['name_ru'] = df_iata['name_en'].progress_apply(translator_error_handle)
print('Успешно!')

Старт перевода названий аэропортов...


100%|██████████| 6072/6072 [52:24<00:00,  1.93it/s]     

Успешно!


In [14]:
print('Старт перевода городов аэропортов...')
df_iata['city_ru'] = df_iata['city_en'].progress_apply(translator_error_handle)
print('Успешно!')

Старт перевода городов аэропортов...


100%|██████████| 6072/6072 [20:04<00:00,  5.04it/s]

Успешно!


In [15]:
print('Старт перевода стран аэропортов...')
df_iata['country_ru'] = df_iata['country_en'].progress_apply(translator_error_handle)
print('Успешно!')

Старт перевода стран аэропортов...


100%|██████████| 6072/6072 [00:42<00:00, 144.03it/s] 

Успешно!


In [16]:
df_iata.head()

,name_en,city_en,country_en,IATA,name_ru,city_ru,country_ru
0,Goroka Airport,Goroka,Papua New Guinea,GKA,Аэропорт Горока,Горока,Папуа - Новая Гвинея
1,Madang Airport,Madang,Papua New Guinea,MAG,Аэропорт Маданга,Маданг,Папуа - Новая Гвинея
2,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,Аэропорт Маунт-Хаген-Кагамуга,Гора Хаген,Папуа - Новая Гвинея
3,Nadzab Airport,Nadzab,Papua New Guinea,LAE,Аэропорт Надзаб,Надзаб,Папуа - Новая Гвинея
4,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,Международный аэропорт Порт-Морсби Джексонс,Порт-Морсби,Папуа - Новая Гвинея


## Отсутствующие значения после перевода

In [17]:
df_iata.isna().sum()

name_en        0
city_en       39
country_en     0
IATA           0
name_ru        0
city_ru       39
country_ru     0
dtype: int64

In [18]:
df_iata[df_iata['city_en'].isna()].head()

,name_en,city_en,country_en,IATA,name_ru,city_ru,country_ru
5794,King Salman Abdulaziz Airport,NaN,Saudi Arabia,DWD,Аэропорт имени короля Салмана Абдулазиза,NaN,Саудовская Аравия
5795,King Khaled Air Base,NaN,Saudi Arabia,KMX,Военно-воздушная база имени короля Халеда,NaN,Саудовская Аравия
5808,Bislig Airport,NaN,Philippines,BPH,Аэропорт Бислиг,NaN,Филиппины
5809,Mati National Airport,NaN,Philippines,MXI,Национальный аэропорт Мати,NaN,Филиппины
5826,Belaya Gora Airport,NaN,Russia,BGN,Аэропорт Белая Гора,NaN,Россия


Закономерно, что для аэропортов для которых город `city_en` не был указан изначально, перевод `city_ru` тоже будет отсутствовать.

## Добавление таблицы в БД

In [19]:
import sqlite3

conn = sqlite3.connect('../data/flights_info.db')
df_iata.to_sql('airports_info', conn, if_exists='replace', index=False)

df_loaded = pd.read_sql('SELECT * FROM airports_info', conn)
conn.close()

In [20]:
df_loaded

,name_en,city_en,country_en,IATA,name_ru,city_ru,country_ru
0,Goroka Airport,Goroka,Papua New Guinea,GKA,Аэропорт Горока,Горока,Папуа - Новая Гвинея
1,Madang Airport,Madang,Papua New Guinea,MAG,Аэропорт Маданга,Маданг,Папуа - Новая Гвинея
2,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,Аэропорт Маунт-Хаген-Кагамуга,Гора Хаген,Папуа - Новая Гвинея
3,Nadzab Airport,Nadzab,Papua New Guinea,LAE,Аэропорт Надзаб,Надзаб,Папуа - Новая Гвинея
4,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,Международный аэропорт Порт-Морсби Джексонс,Порт-Морсби,Папуа - Новая Гвинея
...,...,...,...,...,...,...,...
6067,Bilogai-Sugapa Airport,Sugapa-Papua Island,Indonesia,UGU,Аэропорт Билогай-Сугапа,Сугапа-остров Папуа,Индонезия
6068,Ramon Airport,Eilat,Israel,ETM,Аэропорт Рамон,Эйлат,Израиль
6069,Rustaq Airport,Al Masna'ah,Oman,MNH,Аэропорт Рустак,Аль-Масна'а,Оман
6070,Laguindingan Airport,Cagayan de Oro City,Philippines,CGY,Аэропорт Лагуиндинган,Город Кагаян-де-Оро,Филиппины
